In [107]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tamargurabanidze/housepricesdata/sample_submission.csv
/kaggle/input/datasets/tamargurabanidze/housepricesdata/data_description.txt
/kaggle/input/datasets/tamargurabanidze/housepricesdata/train.csv
/kaggle/input/datasets/tamargurabanidze/housepricesdata/test.csv


In [108]:
!pip install mlflow dagshub xgboost

In [109]:

import dagshub
import mlflow

dagshub.init(
    repo_owner="tgura23",
    repo_name="ml_assignment_housePrices",
    mlflow=True
)

import mlflow
mlflow.set_experiment("house_prices_clean_dagshub")

Initialized MLflow to track repo "tgura23/ml_assignment_housePrices"

Repository tgura23/ml_assignment_housePrices initialized!

<Experiment: artifact_location='mlflow-artifacts:/4ecb8f0983ea4e69a05084a89e14e61a', creation_time=1776087711748, experiment_id='4', last_update_time=1776087711748, lifecycle_stage='active', name='house_prices_clean_dagshub', tags={}, trace_location=None, workspace='default'>

In [110]:
model_name = "HousePrice_XGBoost"
model_version = "latest" 

model_uri = f"models:/{model_name}/{model_version}"
loaded_model = mlflow.sklearn.load_model(model_uri)

In [111]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "test" in file.lower():
            print(os.path.join(root, file))

/kaggle/input/datasets/tamargurabanidze/housepricesdata/test.csv


In [112]:
import pandas as pd
test_df = pd.read_csv('/kaggle/input/datasets/tamargurabanidze/housepricesdata/test.csv')
test_ids = test_df['Id']

In [113]:
test_df["HouseAge"] = test_df["YrSold"] - test_df["YearBuilt"]
test_df["RemodAge"] = test_df["YrSold"] - test_df["YearRemodAdd"]

test_df["TotalSF"] = (
    test_df["1stFlrSF"] +
    test_df["2ndFlrSF"] +
    test_df["TotalBsmtSF"]
)

test_df["TotalBath"] = (
    test_df["FullBath"] +
    0.5 * test_df["HalfBath"] +
    test_df["BsmtFullBath"] +
    0.5 * test_df["BsmtHalfBath"]
)

test_df["TotalPorch"] = (
    test_df["OpenPorchSF"] +
    test_df["EnclosedPorch"] +
    test_df["3SsnPorch"] +
    test_df["ScreenPorch"]
)

import numpy as np

test_df["LogLotArea"] = np.log1p(test_df["LotArea"])
test_df["LogGrLivArea"] = np.log1p(test_df["GrLivArea"])

test_df["QualityScore"] = test_df["OverallQual"] * test_df["GrLivArea"]

In [114]:
qual_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

qual_cols = [
    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
    "HeatingQC", "KitchenQual", "FireplaceQu",
    "GarageQual", "GarageCond"
]

for col in qual_cols:
    test_df[col] = test_df[col].map(qual_map).fillna(0)

In [115]:
ord_cols_maps = {
    "BsmtExposure": {"None":0, "No":1, "Mn":2, "Av":3, "Gd":4},
    "BsmtFinType1": {"None":0, "Unf":1, "LwQ":2, "Rec":3, "BLQ":4, "ALQ":5, "GLQ":6},
    "BsmtFinType2": {"None":0, "Unf":1, "LwQ":2, "Rec":3, "BLQ":4, "ALQ":5, "GLQ":6},
    "GarageFinish": {"None":0, "Unf":1, "RFn":2, "Fin":3},
    "Fence": {"None":0, "MnWw":1, "GdWo":2, "MnPrv":3, "GdPrv":4},
    "LotShape": {"IR3":0, "IR2":1, "IR1":2, "Reg":3},
    "LandSlope": {"Sev":0, "Mod":1, "Gtl":2},
    "PavedDrive": {"N":0, "P":1, "Y":2}
}

for col, mapping in ord_cols_maps.items():
    test_df[col] = test_df[col].map(mapping).fillna(0)

In [116]:
test_df["CentralAir"] = test_df["CentralAir"].map({"N":0, "Y":1})
test_df["Street"] = test_df["Street"].map({"Grvl":0, "Pave":1})

In [117]:
num_cols = test_df.select_dtypes(include=["int64", "float64"]).columns

test_df["LotFrontage"] = test_df["LotFrontage"].fillna(test_df["LotFrontage"].median())
test_df["GarageYrBlt"] = test_df["GarageYrBlt"].fillna(0)
test_df["MasVnrArea"] = test_df["MasVnrArea"].fillna(0)

# everything else safe fallback
test_df[num_cols] = test_df[num_cols].fillna(0)

In [118]:
# ONE-HOT ENCODING (must match training)
woe_cols = [
    'MSZoning',
    'Alley',
    'LandContour',
    'Utilities',
    'LotConfig',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'Electrical',
    'Functional',
    'GarageType',
    'SaleType',
    'SaleCondition'
]

test_df = pd.get_dummies(test_df, columns=woe_cols, drop_first=True)

In [119]:
expected_features = [
    'QualityScore',
    'Neighborhood',
    'TotalBsmtSF',
    'OverallQual',
    'BsmtFinSF1',
    '2ndFlrSF',
    '1stFlrSF',
    'BsmtQual',
    'GarageArea',
    'LotFrontage',
    'YearBuilt',
    'RemodAge',
    'GarageYrBlt',
    'LogLotArea',
    'BsmtUnfSF',
    'LotArea',
    'TotalBath',
    'KitchenQual',
    'OpenPorchSF',
    'TotalPorch',
    'WoodDeckSF',
    'BsmtExposure',
    'GarageFinish',
    'MasVnrArea',
    'MSZoning',
    'FireplaceQu',
    'ExterQual',
    'BsmtFinType1',
    'Exterior2nd',
    'SaleCondition',
    'GarageType',
    'TotRmsAbvGrd',
    'OverallCond',
    'CentralAir',
    'Exterior1st',
    'MSSubClass',
    'BedroomAbvGr',
    'LotShape',
    'FullBath',
    'HeatingQC'
]

In [120]:
test_df = test_df.reindex(columns=expected_features, fill_value=0)

In [121]:
predictions = loaded_model.predict(test_df)

In [126]:
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission.to_csv('submission_xgboost.csv', index=False)
print('done')

done
